In [1]:
import numpy as np 
#want to generate U, V randomly. Initialize each entry to be drawn from a normal independently. 
#U has dimensions n x r
n = 100
r = 10
p = 500
U_true = np.random.normal(0,1,size=(n,r))
V_true = np.random.normal(0,1,size=(p,r))
Theta_true = np.matmul(U_true,V_true.T) #shape: (n,p)



In [ ]:
P_true = 1/(1+np.exp(-Theta_true)) #shape: (n,p)
Y_obs = np.random.binomial(1,P_true)
#let g1,g2 have n/2 samples each. Let g1 unique features be the 
# first floor(p/3) cols, g2 unique features be the second floor(p/3) cols, shared features are the last p-2*floor(p/3) cols
g1_unique = Y_obs[:n // 2, :p // 3]
g1_shared = Y_obs[:n // 2, 2*(p//3):]
g2_unique = Y_obs[n // 2:, p // 3:2*(p//3)]
g2_shared = Y_obs[n // 2:,2*(p//3):]
Y_g1 = np.concatenate([g1_unique,g1_shared],axis= 1)
Y_g2 = np.concatenate([g2_unique,g2_shared],axis=1)
Y_shared = np.concatenate([g1_shared,g2_shared],axis=0)


In [3]:
U_g1=np.random.normal(0,0.01,size=(n // 2,r))
V_g1 = np.random.normal(0,0.01,size=(p - p // 3,r))
U_g2 = np.random.normal(0,0.01,size=(n - n//2,r))
V_g2 = np.random.normal(0,0.01,size=(p - p // 3,r))


In [4]:
def nll(U,V,Y):
    Theta = np.matmul(U,V.T)
    inside = np.logaddexp(np.zeros(Theta.shape),Theta)
    summand = inside - Y*Theta
    return np.sum(summand)
def sigmoid(x):
    return 1/(1+np.exp(-x))
def s1_fit(Y_g,lr,tol,max_iters,U_g,V_g):
    iters = 0
    prev_nll = float('inf')
    curr_nll = nll(U_g,V_g,Y_g)
    while abs(curr_nll-prev_nll) > tol and iters <= max_iters:
        R_g = sigmoid(np.matmul(U_g,V_g.T))-Y_g
        u_update = lr*np.matmul(R_g,V_g)
        v_update = lr*np.matmul(R_g.T,U_g)
        U_g,V_g = U_g-u_update,V_g-v_update
        prev_nll = curr_nll
        curr_nll = nll(U_g,V_g,Y_g)
        iters += 1
    return U_g,V_g


    



In [5]:
U_g1s1,V_g1s1 = s1_fit(Y_g1,0.001,10**-6,10000,U_g1,V_g1)
U_g2s1,V_g2s1=s1_fit(Y_g2,0.001,10**-6,10000,U_g2,V_g2)

In [10]:
Q_g1,_ = np.linalg.qr(U_g1s1,mode="reduced")
Q_g2,_ = np.linalg.qr(U_g2s1,mode="reduced")
print(Q_g1.shape,Q_g2.shape)
Q_hat = np.concatenate([np.concatenate([Q_g1,np.zeros((Q_g1.shape[0],Q_g2.shape[1]))],axis=1),np.concatenate([np.zeros((Q_g2.shape[0],Q_g1.shape[1])),Q_g2],axis=1)],axis=0)
print(Q_hat.shape)

(50, 10) (50, 10)
(100, 20)


In [7]:
proj = np.matmul(Q_hat,Q_hat.T)
U_hat = np.random.normal(0,0.01,size=(n,r))
V_hat = np.random.normal(0,0.01,size=(p-2*(p // 3),r))
def s2_fit(U,V,Y,lr,tol,max_iters,proj):
    U = np.matmul(proj,U)
    iters = 0
    prev_nll = float('inf')
    curr_nll = nll(U,V,Y)
    while abs(curr_nll-prev_nll) >= tol and iters < max_iters:
        R_hat = sigmoid(np.matmul(U,V.T))-Y
        u_update = lr*np.matmul(R_hat,V)
        v_update = lr*np.matmul(R_hat.T,U)
        U,V = U - u_update,V - v_update
        U = np.matmul(proj,U)
        iters += 1
        prev_nll = curr_nll
        curr_nll = nll(U,V,Y)
    return U,V
U_final,V_final = s2_fit(U_hat,V_hat,Y_shared,0.01,10**-6,10000,proj)
    
    

In [9]:
def subspace_error(U_true,U_est):
    Q_est, _ = np.linalg.qr(U_est,mode="reduced")
    Q_true,_ = np.linalg.qr(U_true,mode="reduced")
    proj_est = np.matmul(Q_est,Q_est.T)
    proj_true = np.matmul(Q_true,Q_true.T)
    r = U_true.shape[1]
    return np.linalg.norm(proj_est-proj_true,ord= "fro")/np.sqrt(2*r)
print(subspace_error(U_true,U_final))

0.5088404388100951


In [12]:
U_baseline = np.random.normal(0,0.01,size=(n,r))
V_baseline = np.random.normal(0,0.01,size=(p-2*(p // 3),r))
baseline_proj = np.identity(n)
U_final_baseline,V_final_baseline = s2_fit(U_baseline,V_baseline,Y_shared,0.01,10**-6,10000,baseline_proj)
print(subspace_error(U_true,U_final_baseline))

0.7416276056986038


In [14]:
def random_sim(seed,n,p,r,gsize1_samples,gsize1_features,gsize2_features):
    np.random.seed(seed)
    U_true = np.random.normal(0,1,size=(n,r))
    V_true = np.random.normal(0,1,size=(p,r))
    Theta_true = np.matmul(U_true,V_true.T) #shape: (n,p)
    P_true = 1/(1+np.exp(-Theta_true)) #shape: (n,p)
    Y_obs = np.random.binomial(1,P_true)
    #let g1,g2 have n/2 samples each. Let g1 unique features be the 
    # first floor(p/3) cols, g2 unique features be the second floor(p/3) cols, shared features are the last p-2*floor(p/3) cols
    g1_unique = Y_obs[:gsize1_samples, :gsize1_features]
    g1_shared = Y_obs[:gsize1_samples, gsize1_features+gsize2_features:]
    g2_unique = Y_obs[gsize1_samples:, gsize1_features:gsize1_features+gsize2_features]
    g2_shared = Y_obs[gsize1_samples:,gsize1_features+gsize2_features:]
    Y_g1 = np.concatenate([g1_unique,g1_shared],axis= 1)
    Y_g2 = np.concatenate([g2_unique,g2_shared],axis=1)
    Y_shared = np.concatenate([g1_shared,g2_shared],axis=0)
    U_g1=np.random.normal(0,0.01,size=(gsize1_samples,r))
    V_g1 = np.random.normal(0,0.01,size=(p - gsize2_features,r))
    U_g2 = np.random.normal(0,0.01,size=(n - gsize1_samples,r))
    V_g2 = np.random.normal(0,0.01,size=(p - gsize1_features,r))
    U_g1s1,V_g1s1 = s1_fit(Y_g1,0.001,10**-6,10000,U_g1,V_g1)
    U_g2s1,V_g2s1=s1_fit(Y_g2,0.001,10**-6,10000,U_g2,V_g2)
    Q_g1,_ = np.linalg.qr(U_g1s1,mode="reduced")
    Q_g2,_ = np.linalg.qr(U_g2s1,mode="reduced")
    Q_hat = np.concatenate([np.concatenate([Q_g1,np.zeros((Q_g1.shape[0],Q_g2.shape[1]))],axis=1),np.concatenate([np.zeros((Q_g2.shape[0],Q_g1.shape[1])),Q_g2],axis=1)],axis=0)
    proj = np.matmul(Q_hat,Q_hat.T)
    U_hat = np.random.normal(0,0.01,size=(n,r))
    V_hat = np.random.normal(0,0.01,size=(p-gsize1_features-gsize2_features,r))
    U_final,V_final = s2_fit(U_hat,V_hat,Y_shared,0.01,10**-6,10000,proj)
    print(subspace_error(U_true,U_final))
    U_baseline = np.random.normal(0,0.01,size=(n,r))
    V_baseline = np.random.normal(0,0.01,size=(p-gsize1_features-gsize2_features,r))
    baseline_proj = np.identity(n)
    U_final_baseline,V_final_baseline = s2_fit(U_baseline,V_baseline,Y_shared,0.01,10**-6,10000,baseline_proj)
    print(subspace_error(U_true,U_final_baseline))
    test,base = subspace_error(U_true,U_final),subspace_error(U_true,U_final_baseline)
    return test,base
results = np.array([random_sim(seed, 100, 500, 10, 50, 500 // 3, 500 // 3) for seed in range(50)])
test_mean = results[:,0].mean()    
base_mean = results[:,1].mean()
print(test_mean,base_mean)

0.465281257218806
0.7137069165932869
0.49598988869209376


/var/folders/k8/zzx8k6051wzd84__vpwc555h0000gn/T/ipykernel_25967/3471261050.py:7: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-x))


0.7367599299155505
0.5412693919004743
0.6904144240635521
0.46286008253175365
0.7336561613358809
0.6563572332285567
0.7034276538594426
0.518782497136782
0.7088148526776115
0.5521092355491604
0.7239610512230688
0.5472462367231791
0.7302668831807645
0.5797849304106097
0.7161490048675877
0.4840672549347396
0.7326961049428419
0.5069687416047964
0.7627850867529534
0.48800365095940845
0.7247419315195274
0.5220776548560585
0.6862101027110243
0.48885692471925457
0.7334599384292954
0.5204125884753092
0.68024849045273
0.5760243902743649
0.7327874011255545
0.5320292045730625
0.7330800374071441
0.5222323883805735
0.6836041020961955
0.5577474594644564
0.7053700485437661
0.5505193960352962
0.6930469024709193
0.5848912368023891
0.7454481456814819
0.5040804750542413
0.7439497496643741
0.5377153632489584
0.7222842945543827
0.5225489780465854
0.7571983092293699
0.5719423971337825
0.7295030237294541
0.5361337111449628
0.6936462551493406
0.6055720075304549
0.674586379126443
0.45171635280094447
0.7623418744

In [18]:
improvements = results[:,0]-results[:,1]
print(improvements.std(ddof=1))
print(f"Number won: {np.mean(improvements < 0)}")

0.062427414761122776
Number won: 1.0
